# 03 - Chest X-ray: Evaluation and Grad-CAM


> **Academic prototype.** This notebook is part of a university final project.
> The models here are **not** medical devices, are **not** validated on clinical
> data, and must **never** be used to diagnose, screen or triage real patients.
> See `docs/ETHICS.md`.


**Goal:** measure the trained models properly and look at *why* they predict what
they predict.

Protocol (in this order, deliberately):

1. thresholds are selected on the **validation** set;
2. those frozen thresholds are applied to the **test** set;
3. ROC-AUC, PR-AUC, precision, recall/sensitivity, specificity, F1 and confusion
   matrices are computed on test;
4. bootstrap confidence intervals quantify the sampling noise;
5. Grad-CAM shows which regions drove the predictions.

Accuracy is reported, but only as a secondary number - see `docs/METRICS.md`.

**Prerequisite:** checkpoints from notebook 02.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

%load_ext autoreload
%autoreload 2

print("Project root:", PROJECT_ROOT)

In [ ]:
import numpy as np
import pandas as pd
import torch

from src.common import get_device, load_config, seed_everything
from src.common.errors import DataNotFoundError
from src.common.io_utils import save_csv, save_figure
from src.common.viz import set_plot_style
from src.classification import (
    build_dataloaders, build_model, compare_runs, evaluate_classifier, load_checkpoint,
)

set_plot_style()
pd.set_option("display.width", 160)

manifest = pd.read_csv(PROJECT_ROOT / "data/processed/xray_manifest.csv")
device = get_device("auto")
print(f"{len(manifest)} images | device={device}")

## Load the trained models

The checkpoints saved by notebook 02. If a path is missing, re-run notebook 02 - nothing here trains.

In [ ]:
def load_run(config_name):
    cfg = load_config(config_name)
    seed_everything(cfg.get("seed", 42), deterministic=cfg.get("deterministic", True))
    run_name = cfg.get("run_name")
    ckpt = PROJECT_ROOT / cfg.get("output.root") / run_name / "models" / f"{run_name}_best.pt"
    if not ckpt.exists():
        raise DataNotFoundError(
            f"Checkpoint not found: {ckpt}\n"
            f"  Fix: run 02_xray_training.ipynb to train '{run_name}' first."
        )
    model = build_model(cfg, num_labels=len(cfg.data.labels))
    model = load_checkpoint(ckpt, model, device)
    loaders = build_dataloaders(manifest, cfg, splits=("val", "test"))
    return cfg, model, loaders

cfg_base, model_base, loaders_base = load_run("xray_baseline.yaml")
print()
cfg_tl, model_tl, loaders_tl = load_run("xray_transfer.yaml")

LABELS = list(cfg_tl.data.labels)

## 1. Threshold selection on validation, then test evaluation

`evaluate_classifier` runs the whole protocol and writes every artefact to
`outputs/classification/<run>/`:

| file | content |
|---|---|
| `metrics/thresholds_from_validation.csv` | chosen threshold per label and why |
| `metrics/metrics_test.csv` / `.json` | the reportable test metrics |
| `metrics/metrics_test_bootstrap_ci.csv` | bootstrap confidence intervals |
| `predictions/test_predictions.csv` | per-image probabilities and decisions |
| `figures/*.png` | ROC, PR, confusion matrices, threshold sweeps |

The threshold strategy comes from `eval.threshold_strategy` in the config
(`f1`, `youden` or `min_recall`). Justify your choice in the report: in a
screening context, missing a positive case is usually worse than a false alarm,
which argues for `min_recall`.

In [ ]:
results_base = evaluate_classifier(model_base, loaders_base, cfg_base, device)

In [ ]:
results_tl = evaluate_classifier(model_tl, loaders_tl, cfg_tl, device)

## 2. Head-to-head comparison

The headline table for the report. Accuracy is deliberately last and labelled as secondary.

In [ ]:
comparison = compare_runs([results_base, results_tl], metric="roc_auc")
save_csv(comparison, PROJECT_ROOT / "outputs/classification/model_comparison_test.csv")
comparison

## 3. Detailed metrics for the better model

In [ ]:
best = results_tl if (comparison.iloc[0]["run"] == results_tl["run_name"]) else results_base
best_cfg = cfg_tl if best is results_tl else cfg_base
best_model = model_tl if best is results_tl else model_base

print(f"Best run by test ROC-AUC: {best['run_name']}\n")
best["test_metrics"].round(4)

### Confidence intervals

Percentile bootstrap over the test set (1000 resamples by default). A wide interval means the test set is too small to distinguish the models - an honest and reportable finding, not a failure.

In [ ]:
ci = best.get("bootstrap_ci")
if ci is not None and len(ci):
    display_ci = ci.pivot(index="label", columns="metric", values="formatted")
    print("Point estimate [95% CI] on the test set:\n")
    print(display_ci.to_string())
else:
    print("Bootstrap disabled (eval.n_bootstrap = 0).")

### Reading the confusion matrix

- **FN (bottom-left)** = patients with the finding the model missed. In a clinical framing this is the most costly error.
- **FP (top-right)** = false alarms, causing unnecessary follow-up.

The balance between them is set by the threshold chosen in step 1 - it is a decision, not a model property.

In [ ]:
from src.classification.metrics import plot_confusion_matrix

for i, label in enumerate(LABELS):
    thr = best["thresholds"][label]
    y_true = best["test_true"][:, i]
    y_pred = (best["test_prob"][:, i] >= thr).astype(int)
    fig = plot_confusion_matrix(y_true, y_pred, label_name=f"{label} (threshold {thr:.3f})")
    fig.show()

### Error analysis

The highest-confidence mistakes. Open a few of these images: the cause is often obvious (wrong view, poor exposure, a label error in the dataset) and it makes a much stronger discussion section than the metrics alone.

In [ ]:
predictions = best["predictions"]
label = LABELS[0]

false_negatives = predictions[(predictions[f"true_{label}"] == 1) &
                              (predictions[f"pred_{label}"] == 0)].nsmallest(5, f"prob_{label}")
false_positives = predictions[(predictions[f"true_{label}"] == 0) &
                              (predictions[f"pred_{label}"] == 1)].nlargest(5, f"prob_{label}")

print(f"Most confident FALSE NEGATIVES (missed {label}):")
print(false_negatives[["image_path", f"prob_{label}"]].to_string(index=False))
print(f"\nMost confident FALSE POSITIVES:")
print(false_positives[["image_path", f"prob_{label}"]].to_string(index=False))

## 4. Grad-CAM

Grad-CAM highlights the image regions that most influenced the score for a class.

**How to read it honestly.** A heat map is *region attribution*, not a lesion
outline and not clinical reasoning. Highlights on text markers, tubes, image
borders or the patient's shoulders are common and indicate the model learned a
shortcut correlated with the label rather than the pathology itself. Showing
those cases is more valuable than showing only the ones that look convincing -
include failures (FP/FN) in the panel below.

In [ ]:
from src.classification import gradcam_panel, pick_examples
from src.classification.dataset import XrayDataset
from src.classification.models import get_gradcam_target_layer
from src.classification.transforms import build_transforms

class_index = int(best_cfg.get("gradcam.class_index", 0))
per_group = int(best_cfg.get("gradcam.examples_per_group", 2))
label = LABELS[class_index]
threshold = best["thresholds"][label]

# Rebuild the test dataset in the SAME order the evaluation used (no shuffling).
test_dataset = XrayDataset(
    manifest[manifest["split"] == "test"],
    labels=LABELS,
    transform=build_transforms(int(best_cfg.get("data.image_size", 224)), train=False,
                               pretrained=bool(best_cfg.get("model.pretrained", True))),
    return_path=True,
)

groups = pick_examples(best["test_true"], best["test_prob"], class_index=class_index,
                       threshold=threshold, per_group=per_group)
print({k: len(v) for k, v in groups.items()})

In [ ]:
target_layer = get_gradcam_target_layer(best_model, str(best_cfg.get("model.name")))
figures_dir = best["dirs"]["figures"]

for group_name, indices in groups.items():
    if not indices:
        print(f"[info] no {group_name} examples in the test set - skipping")
        continue
    fig = gradcam_panel(
        best_model, test_dataset, indices, target_layer, LABELS, device,
        class_index=class_index,
        pretrained=bool(best_cfg.get("model.pretrained", True)),
        alpha=float(best_cfg.get("gradcam.alpha", 0.4)),
        thresholds=best["thresholds"],
    )
    save_figure(fig, figures_dir / f"gradcam_{group_name}.png", close=False)

---

## Notes for the report

Write these up from the numbers above - never from memory or expectation:

- test ROC-AUC and PR-AUC with confidence intervals, for both models;
- sensitivity **and** specificity at the chosen threshold, plus the reasoning for
  that threshold;
- how many positives were missed (FN) and what that would mean in practice;
- whether the confidence intervals of the two models overlap (if they do, you
  cannot claim one is better);
- what the Grad-CAM maps actually attend to, including the failures.

**Limitations to state explicitly:** single dataset, single site, no external
validation, no clinical validation, image-level bootstrap slightly optimistic
when patients contribute several images.

### Screenshots for the report
- The model comparison table
- ROC and PR curves
- Confusion matrix at the chosen threshold
- Threshold-sweep figure (justifies the operating point)
- Grad-CAM panels - include at least one FP or FN

**Next:** `04_mri_eda_and_preparation.ipynb`